# Feature selection — forecast t+1 (`prod_gas`), set recursion-safe

Notebook análogo a `05_feature_selection_pet.ipynb`, pero para **gas** (ADR-042).
Mismo protocolo, mismas features, pero calculadas sobre `prod_gas`:

1. **Se parte del set recursion-safe**: se excluyen las features que **no** se pueden
   recalcular en un mes futuro a partir de la propia trayectoria de `prod_gas` (§1.1).
2. **Primero se tunean los hiperparámetros** (CV temporal, expanding window).
3. **Después se hace feature selection** sobre el campeón: *permutation importance*
   en **val** y curva top-k para el set más chico que no pierde RMSE.

**Simetría clave con petróleo:** en el notebook de petróleo se excluía `prod_gas`
(serie medida que no forecasteamos); acá, por lo mismo, se excluye **`prod_pet`**, y
el motor autorregresivo pasa a ser `prod_gas` + `prod_gas_*`.

**Anti-leakage (ADR-028):** se tunea/entrena en `train`, se rankea y selecciona en
`val`; **`test` queda intacto**. La lógica reutilizable vive en `ml/modeling.py` y
`ml/selection.py`; acá solo se orquesta.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml import modeling, selection

TARGET = "prod_gas"
pd.set_option("display.width", 120)

## 1. Dataset y matriz de features

`build_feature_matrix(target='prod_gas')` arma el dataset del **universo gasífero**
(pozos con `prod_gas` > 0 en train) con las features de ingeniería calculadas sobre
`prod_gas` (`prod_gas_roll3`, `prod_gas_lag2`, …) y el target `y_next` = `prod_gas`
del mes t+1. Devuelve las features **crudas** (el one-hot lo hace el `Pipeline`).

In [2]:
ds_enc, feature_cols = modeling.build_feature_matrix(target=TARGET)

print(f"Filas: {len(ds_enc):,}  |  Features totales: {len(feature_cols)}")
print(ds_enc["split"].value_counts().reindex(["train", "val", "test"]).to_string())

Filas: 364,824  |  Features totales: 40
split
train    255321
val       54675
test      54828


## 1.1 Excluir las features NO recursion-safe

Para el forecast recursivo de gas, cada feature de un mes futuro tiene que poder
calcularse desde **la trayectoria del propio `prod_gas`** (+ estáticas + calendario).
Se excluyen las que no cumplen esto **antes** de tunear y seleccionar:

| Feature | Por qué NO es recursion-safe |
|---|---|
| `prod_vecinos_mean` | Cross-well: depende de la producción de **otros** pozos en el mes t. |
| `water_cut` | Necesita `prod_agua` del mes futuro, que no forecasteamos. |
| `prod_pet` | Serie **cruzada** de petróleo: en el mes futuro no la tenemos (este modelo no forecastea petróleo). |
| `prod_agua` | Otra serie medida del pozo; no la forecasteamos. |
| `tef` | Variable operativa del mes futuro, desconocida al predecir. |

Se **mantiene** `prod_gas` (último valor del target = persistencia) y sus derivadas
autorregresivas (`prod_gas_roll3/delta1/lag12/acum6/lag2/lag3/roll6/acum12/delta3/`
`ratio1/std3/cummax/frac_peak/meses_desde_pico`), más `well_age_months`, las estáticas
(`profundidad`, coords, categóricas) y `mes`.

> Nota la simetría: acá se excluye **`prod_pet`** (y se conserva `prod_gas`); en el
> notebook de petróleo era al revés.

In [3]:
# No recursion-safe para el modelo de GAS: OJO -> se excluye prod_pet (no prod_gas).
NO_RECURSION_SAFE = [
    "prod_vecinos_mean",  # cross-well: depende de OTROS pozos
    "water_cut",          # depende de prod_agua(t+1)
    "prod_pet",           # serie cruzada de petroleo, no la forecasteamos aca
    "prod_agua",          # otra serie medida, no la forecasteamos
    "tef",                # variable operativa del mes futuro, desconocida
]

feature_cols = [c for c in feature_cols if c not in NO_RECURSION_SAFE]
print(f"Excluidas: {NO_RECURSION_SAFE}")
print(f"Features recursion-safe: {len(feature_cols)}")
print(feature_cols)

Excluidas: ['prod_vecinos_mean', 'water_cut', 'prod_pet', 'prod_agua', 'tef']
Features recursion-safe: 35
['prod_gas', 'profundidad', 'coordenadax', 'coordenaday', 'mes', 'prod_gas_roll3', 'prod_gas_delta1', 'prod_gas_lag12', 'prod_gas_acum6', 'produjo_mes_pasado', 'prod_gas_lag2', 'prod_gas_lag3', 'prod_gas_roll6', 'prod_gas_acum12', 'prod_gas_delta3', 'prod_gas_ratio1', 'prod_gas_std3', 'prod_gas_cummax', 'prod_gas_frac_peak', 'prod_gas_meses_desde_pico', 'well_age_months', 'tipoextraccion', 'tipoestado', 'tipopozo', 'empresa', 'formprod', 'formacion', 'areapermisoconcesion', 'areayacimiento', 'cuenca', 'provincia', 'proyecto', 'clasificacion', 'subclasificacion', 'sub_tipo_recurso']


## 2. Split train / val

`X` queda como DataFrame con las columnas crudas; el `Pipeline` hace el
one-hot / imputación / escalado ajustando **solo en train**. Guardamos
`X_train`/`X_val` porque los reusa la feature selection.

In [4]:
X_train, y_train, X_val, y_val = modeling.split_train_val(ds_enc, feature_cols)
print("train:", X_train.shape, " | val:", X_val.shape)

train: (255321, 35)  | val: (54675, 35)


## 3. Búsqueda de hiperparámetros (primero) — CV temporal

`tune_all` corre un **random search** con **CV temporal (expanding window por mes)**:
cada fold valida en meses posteriores a su train → sin leakage futuro→pasado. El
preprocesamiento va dentro del `Pipeline` (se reajusta por fold). Modelos: **Ridge**,
**Random Forest**, **XGBoost**.

> ⚠️ Entrena muchos modelos: puede tardar varios minutos. Cada modelo muestra su
> barra de progreso con el mejor RMSE de CV hasta el momento.

In [ ]:
tabla_cv, searches = modeling.tune_all(ds_enc, feature_cols, n_splits=4)
tabla_cv

ridge (20 configs):   0%|          | 0/20 [00:00<?, ?it/s]

random_forest (20 configs):   0%|          | 0/20 [00:00<?, ?it/s]

### 3.1 Evaluación de los tuneados en val + elección del campeón

Cada `best_estimator_` se evalúa sobre **val**. El **campeón** es el de menor
`val_rmse` (excluyendo la persistencia, que para gas es `ŷ(t+1) = prod_gas(t)`).

In [ ]:
comparacion_tuned = modeling.eval_searches_on_val(searches, ds_enc, feature_cols, target=TARGET)
comparacion_tuned.round(3)

In [ ]:
nombre_campeon = min(
    searches,
    key=lambda n: modeling.evaluate(y_val, searches[n].best_estimator_.predict(X_val))["val_rmse"],
)
champion = searches[nombre_campeon].best_estimator_
print(f"Campeón (menor val_rmse): {nombre_campeon}")
print("Hiperparámetros:", searches[nombre_campeon].best_params_)

## 4. Feature selection (sobre el campeón)

### 4.1 Ranking por *permutation importance* en val

Se permuta cada feature en val (`n_repeats` veces) y se mide cuánto sube el RMSE.
Como el `Pipeline` hace el one-hot adentro, se permuta a nivel de **feature cruda**.
Importancia en **m³** = cuánto empeora el RMSE al romper la feature (> 0 = útil).

In [ ]:
ranking = selection.permutation_ranking(champion, X_val, y_val, n_repeats=10)
ranking.round(2)

Visualización del ranking (las 20 features más importantes; barra = aumento de RMSE
al permutar, con su desvío entre repeticiones):

In [ ]:
import matplotlib.pyplot as plt

top = ranking.head(20).iloc[::-1]  # la más importante, arriba
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["feature"], top["imp_mean"], xerr=top["imp_std"], color="#DD8452")
ax.axvline(0, color="0.6", lw=0.8)
ax.set_xlabel("Importancia = ↑ RMSE de val al permutar (m³)")
ax.set_title(f"Permutation importance en val — {nombre_campeon} (gas)")
fig.tight_layout()
plt.show()

### 4.2 Curva top-k: RMSE en val según nº de features

Reentrenamos el campeón (mismos hiperparámetros) con solo las **top-k** features del
ranking. Buscamos la **rodilla**: a partir de cierto `k`, sumar features ya no baja
el RMSE.

In [ ]:
curva = selection.evaluate_topk(ranking, champion, X_train, y_train, X_val, y_val)
curva.round(3)

### 4.3 Set seleccionado y comparación completo vs reducido

`select_features` se queda con las features de importancia **> 0**. Comparamos el
campeón **completo** vs **reducido** (idénticos hiperparámetros) en val.

In [ ]:
seleccionadas = selection.select_features(ranking)  # imp_mean > 0
print(f"Seleccionadas ({len(seleccionadas)}/{len(feature_cols)}):")
print(seleccionadas)

In [ ]:
_, m_full = selection.refit_and_eval(champion, feature_cols, X_train, y_train, X_val, y_val)
_, m_red = selection.refit_and_eval(champion, seleccionadas, X_train, y_train, X_val, y_val)

pd.DataFrame([
    {"modelo": f"{nombre_campeon} (completo)", "n_features": len(feature_cols), **m_full},
    {"modelo": f"{nombre_campeon} (reducido)", "n_features": len(seleccionadas), **m_red},
]).round(3)

## 5. Conclusión / próximos pasos

- Corrida sobre el set **recursion-safe** de gas (sin `prod_vecinos_mean`,
  `water_cut`, `prod_pet`, `prod_agua`, `tef`). El ranking dice cuáles de las que
  quedan aportan de verdad.
- Comparar con el ranking de **petróleo** (notebook 05): esperamos un patrón parecido
  (dominan `prod_gas` y sus derivadas de nivel/tendencia; entre las estáticas, la de
  yacimiento/área como ancla de **cold-start**).
- Documentar el set final de gas en el **ADR-043** (aplica la misma selección con
  features `prod_gas_*`), y considerar la robustez a pozos sin historial igual que en
  petróleo (las autorregresivas quedan NaN → 0 + flag; sostienen las estáticas).
- Opcional: **re-tunear** con el set reducido y evaluar en **val y test**.